![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E01 - PromptTemplate: del string al componente (Resolution)

## BLOQUE 1 — ¿Qué es un PromptTemplate y por qué existe?

### El problema de base

Un LLM recibe texto. Ese texto hay que armarlo de alguna forma. En M3L1 lo hacíamos con f-strings:

```python
prompt = f"Contexto: {contexto}\nPregunta: {pregunta}"
```

Eso funciona, pero:
- Las variables están **implícitas** (hay que leer el string para saber qué espera)
- No se puede **inspeccionar** el prompt sin ejecutarlo
- Si el prompt cambia, hay que buscarlo en **todo el código**
- No hay **validación**: si olvidás pasar `{context}`, el f-string explota en runtime

`ChatPromptTemplate` resuelve todo eso convirtiendo el prompt en un **objeto** con variables explícitas, formato declarativo y métodos de inspección.

### Este notebook NO necesita API key

Solo construimos, formateamos, inspeccionamos y debugueamos prompts. No llamamos al modelo.

## BLOQUE 2 — Tipos de PromptTemplate en LangChain

LangChain tiene varias formas de construir prompts. Cada una sirve para un caso distinto.

### 1. `ChatPromptTemplate.from_messages()` — La más común (RECOMENDADA)

Trabaja con lista de mensajes con roles explícitos (`system`, `human`, `ai`, `placeholder`).

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de RRHH."),
    ("human", "Contexto:\n{context}\n\nPregunta:\n{question}"),
])
```

### 2. `ChatPromptTemplate.from_template()` — Prompt único como string

Cuando solo necesitás un mensaje human (sin system).

```python
prompt = ChatPromptTemplate.from_template("Traduce al inglés: {texto}")
```

### 3. `PromptTemplate` — Para LLMs no-chat (legacy)

La versión original de LangChain para modelos de completado (text-davinci, etc.).

```python
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("Resume: {texto}")
```

### 4. `MessagesPlaceholder` — Para inyectar listas dinámicas de mensajes

Esencial para memoria conversacional. Permite insertar N mensajes en una posición fija.

```python
from langchain_core.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente."),
    MessagesPlaceholder(variable_name="historial"),
    ("human", "{pregunta}"),
])
```

Esto permite inyectar 0, 1, 10 mensajes de historial en la misma posición.

## BLOQUE 3 — El problema: concatenación manual de strings

Arranquemos viendo el enfoque "táctico" que traemos de M3L1 y entendamos por qué duele.

In [ ]:
context = "La politica de vacaciones es de 15 dias por ano."
question = "Cuantos dias de vacaciones tengo?"

# --- Version 1: f-string simple ---
prompt_v1 = f"Contexto: {context}\nPregunta: {question}"
print("Version 1 (f-string simple):")
print(prompt_v1)
print()

# --- Version 2: concatenacion con + ---
system_msg = "Eres un asistente de RRHH. Responde solo con el contexto dado."
prompt_v2 = system_msg + "\n\n" + "Contexto: " + context + "\n\nPregunta: " + question
print("Version 2 (concatenacion):")
print(prompt_v2)

### Los 5 problemas del string manual

| # | Problema | Ejemplo |
|---|---|---|
| 1 | **Variables implícitas** | `{context}` y `{question}` existen solo en la cabeza del programador |
| 2 | **Formato mezclado con lógica** | El armado del prompt está dentro de la misma función que llama al LLM |
| 3 | **No inspeccionable** | No podés preguntarle al código "¿qué variables espera este prompt?" |
| 4 | **Duplicación** | Si el mismo prompt se usa en 5 funciones y cambiás algo, tenés que acordarte de las 5 |
| 5 | **Error silencioso** | Si escribís mal `{contex}` en vez de `{context}`, te enterás cuando explota |

> **Regla práctica**: si tenés más de 2 f-strings para prompts en tu código, ya necesitás `ChatPromptTemplate`.

## BLOQUE 4 — La solución: ChatPromptTemplate paso a paso

### Import

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

### Creación del template

La función `build_rag_prompt()` devuelve un `ChatPromptTemplate`.

**Estructura**:
- `system`: define el rol y las reglas del asistente
- `human`: el mensaje del usuario, con `{context}` y `{question}` como variables

**Formato**: cada entrada es un tuple `(rol, contenido)` donde `rol` puede ser:
- `"system"` — instrucciones fijas para el modelo
- `"human"` — lo que escribe el usuario (con variables)
- `"ai"` — respuestas previas (para few-shot)
- `MessagesPlaceholder(...)` — inyección dinámica de mensajes

In [ ]:
def build_rag_prompt() -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres un asistente de RRHH. "
            "Responde usando solo el contexto proporcionado. "
            "Si la respuesta no esta en el contexto, indica que no tienes informacion suficiente."
        ),
        (
            "human",
            "Contexto:\n{context}\n\nPregunta:\n{question}"
        )
    ])


prompt = build_rag_prompt()
print(f"Tipo del prompt: {type(prompt)}")
print(f"Variables requeridas: {prompt.input_variables}")

### Debugging del prompt: `format_messages()`

El método más importante para debugging es **`.format_messages()`**. Te muestra EXACTAMENTE lo que va a recibir el LLM, sin tener que llamar al modelo.

Esto es clave para:
- Verificar que las variables se interpolan correctamente
- Revisar el formato del system message
- Confirmar el orden de los mensajes
- Compartir el prompt con el equipo sin ejecutar nada

In [ ]:
context_test = "La politica de vacaciones es de 15 dias por ano."
question_test = "Cuantos dias de vacaciones tengo?"

messages = prompt.format_messages(context=context_test, question=question_test)

print("=== DEBUG: mensajes formateados ===")
for i, msg in enumerate(messages):
    print(f"--- Mensaje {i} (rol: {msg.type}) ---")
    print(msg.content)
    print(f"--- Metadata: {msg.response_metadata}")
    print()
print("=== FIN DEBUG ===")

### El template es reutilizable (mismos datos, distinto contenido)

El template funciona como una **función**: mismo formato, distintos datos.

In [ ]:
context_rrhh = "Los empleados pueden tomar hasta 3 dias de licencia por enfermedad sin certificado."
question_rrhh = "Necesito tomar un dia por enfermedad. Que necesito presentar?"

messages_rrhh = prompt.format_messages(context=context_rrhh, question=question_rrhh)
for msg in messages_rrhh:
    print(f"[{msg.type.upper()}] {msg.content}")
    print()

print("El mismo template, distintos datos. El formato es siempre consistente.")

## BLOQUE 5 — Inspección profunda del template (debugging avanzado)

Una de las mayores ventajas de `ChatPromptTemplate` es que podés inspeccionar **todo** el objeto programáticamente.

Con un f-string no podés preguntarle "¿cuáles son tus variables?" o "¿cuántos mensajes tenés?".
Con `ChatPromptTemplate` sí.

In [ ]:
print("========== INSPECCION DEL TEMPLATE ==========")
print(f"1. Tipo del objeto: {type(prompt).__name__}")
print(f"2. Variables requeridas: {prompt.input_variables}")
print(f"3. Variables opcionales: {prompt.optional_variables}")
print(f"4. Cantidad de mensajes: {len(prompt.messages)}")
print()
print("--- Detalle de cada mensaje ---")
for i, msg_template in enumerate(prompt.messages):
    print(f"  Mensaje {i}:")
    print(f"    Tipo de template: {type(msg_template).__name__}")
    print(f"    Tipo de mensaje: {msg_template.type}")
    # Cada mensaje internamente tiene un PromptTemplate para su contenido
    if hasattr(msg_template, 'prompt'):
        print(f"    Template interno: {type(msg_template.prompt).__name__}")
        print(f"    Variables del mensaje: {msg_template.prompt.input_variables}")
        print(f"    Template string: {str(msg_template.prompt.template)[:100]}...")
    print()
print("========================================")

### ¿Por qué esto no es posible con un f-string?

```python
# Con f-string:
texto = f"Contexto: {ctx}\nPregunta: {q}"
# No hay forma de preguntar:
# - "¿qué variables espera este string?"
# - "¿cuántos mensajes tiene?"
# - "¿cuál es el system prompt?"

# Con ChatPromptTemplate:
prompt.input_variables      # ['context', 'question']
prompt.messages             # [SystemMessagePromptTemplate, HumanMessagePromptTemplate]
prompt.messages[0].prompt.template  # el texto crudo del system
```

Esto habilita herramientas de debugging, validación automática, documentación generada, etc.

## BLOQUE 6 — Ejemplos avanzados de PromptTemplate

### 6.1 Template con few-shot (mensaje AI de ejemplo)

In [ ]:
# Prompt con un ejemplo de respuesta (few-shot)
prompt_few_shot = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de RRHH."),
    ("human", "Contexto:\n{context}\n\nPregunta:\n{question}"),
    ("ai", "Ejemplo de respuesta: Segun la politica, los empleados tienen derecho a {ejemplo}."),
])

print(f"Prompt con few-shot - variables: {prompt_few_shot.input_variables}")
print()

msgs = prompt_few_shot.format_messages(
    context="Politica: 15 dias de vacaciones.",
    question="Cuantos dias?",
    ejemplo="15 dias"
)
for msg in msgs:
    print(f"[{msg.type.upper()}] {msg.content}")

### 6.2 Template con MessagesPlaceholder (para memoria/historial)

Este es el patrón que se usa en **E04** y **E11** para chat con memoria.

In [ ]:
prompt_con_historial = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente util."),
    MessagesPlaceholder(variable_name="historial"),  # aqui se inyectan los mensajes previos
    ("human", "{pregunta}"),
])

print(f"Variables del prompt con historial: {prompt_con_historial.input_variables}")
print(f"Variables opcionales: {prompt_con_historial.optional_variables}")
print()

# Simular historial con 2 mensajes
historial = [
    HumanMessage(content="Hola, mi nombre es Ana"),
    AIMessage(content="Hola Ana, encantado de conocerte!"),
]

msgs = prompt_con_historial.format_messages(
    historial=historial,
    pregunta="Como me llamo?"
)
for msg in msgs:
    print(f"[{msg.type.upper()}] {msg.content}")

### 6.3 Partial template (valores fijos por defecto)

Cuando parte del prompt es fija y no querés pasarla cada vez.

In [ ]:
prompt_parcial = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente que responde en {idioma}."),
    ("human", "{pregunta}"),
])

# Fijar idioma de una vez
prompt_parcial_fijo = prompt_parcial.partial(idioma="espanol")

print(f"Variables originales: {prompt_parcial.input_variables}")
print(f"Variables despues de partial(): {prompt_parcial_fijo.input_variables}")
print()

msgs = prompt_parcial_fijo.format_messages(pregunta="Hola, como estas?")
for msg in msgs:
    print(f"[{msg.type.upper()}] {msg.content}")

### 6.4 Validación: ¿qué pasa si falta una variable?

In [ ]:
try:
    # Falta la variable 'context'
    prompt.format_messages(question="Solo paso la pregunta?")
    print("NO LANZO ERROR")
except Exception as e:
    print(f"ERROR esperado al faltar 'context':")
    print(f"  Tipo: {type(e).__name__}")
    print(f"  Mensaje: {e}")

print()
print("Con un f-string recibirias un KeyError en production.")
print("Con ChatPromptTemplate, la validacion ocurre en .format_messages().")

### 6.5 Concatenar múltiples prompts

Podés combinar prompts usando `+`.

In [ ]:
base = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente."),
])

saludo = ChatPromptTemplate.from_messages([
    ("human", "Hola, mi nombre es {nombre}."),
])

consulta = ChatPromptTemplate.from_messages([
    ("human", "Quiero saber sobre {tema}."),
])

prompt_combinado = base + saludo + consulta
print(f"Template combinado:")
print(f"  Variables: {prompt_combinado.input_variables}")
print(f"  Mensajes: {len(prompt_combinado.messages)}")
print()

msgs = prompt_combinado.format_messages(nombre="Ana", tema="vacaciones")
for msg in msgs:
    print(f"[{msg.type.upper()}] {msg.content}")

### 6.6 Pipe con `.invoke()` directamente

El template también responde a `.invoke()`, que es lo mismo que `.format_messages()` pero devuelve una lista de mensajes. Esto se usa cuando el template va en una chain LCEL.

In [ ]:
resultado = prompt.invoke({"context": "Texto de prueba.", "question": "Pregunta de prueba?"})
print(f"Tipo del resultado de .invoke(): {type(resultado)}")
print(f"Cantidad de mensajes: {len(resultado)}")
for msg in resultado:
    print(f"  [{msg.type}] {msg.content[:80]}...")

## BLOQUE 7 — Comparación final: f-string vs ChatPromptTemplate

| Aspecto | f-string / string manual | `ChatPromptTemplate` |
|---|---|---|
| **Variables** | Implícitas (hay que leer el string) | Explícitas (`prompt.input_variables`) |
| **Roles** | No existen (todo es texto plano) | `system`, `human`, `ai`, `placeholder` |
| **Inspección** | Imposible | `prompt.messages`, `prompt.input_variables`, etc. |
| **Debugging** | Print del string resultante | `.format_messages()` paso a paso |
| **Reutilización** | Copiar y pegar | El objeto se pasa como parámetro |
| **Validación** | Solo en runtime (KeyError) | En `.format_messages()` antes del LLM |
| **Composición** | No aplica | `prompt1 + prompt2` |
| **Valores parciales** | No | `.partial(variable=valor)` |
| **Historial dinámico** | No | `MessagesPlaceholder(variable_name="historial")` |
| **Documentación** | Ninguna | El objeto se auto-documenta vía inspección |

### ¿Cuándo conviene cada uno?

| Usá f-string cuando... | Usá `ChatPromptTemplate` cuando... |
|---|---|
| Es un script de 1 uso | El prompt se usa en varias funciones |
| El prompt es trivial (1 línea) | El prompt tiene system + human + variables |
| Estás prototipando en una notebook | Estás construyendo un pipeline o agente |
| No hay plan de mantenimiento | Va a producción o lo mantiene un equipo |

## BLOQUE 8 — Checks automáticos (validación del template)

In [ ]:
def run_checks():
    assert prompt is not None
    assert isinstance(prompt, ChatPromptTemplate)
    assert "context" in prompt.input_variables
    assert "question" in prompt.input_variables
    assert len(prompt.messages) >= 2
    msgs = prompt.format_messages(context="Contexto de prueba.", question="Pregunta de prueba?")
    assert len(msgs) >= 2
    assert "Contexto de prueba." in msgs[-1].content
    assert "Pregunta de prueba?" in msgs[-1].content
    print("M3L2 E01 Resolution checks passed")

run_checks()

## Cierre

Hoy aprendiste que:

1. `ChatPromptTemplate` es un **objeto**, no un string — podés inspeccionarlo, validarlo y reutilizarlo
2. Las variables son **explícitas** — el template declara qué necesita
3. El debugging es **posible** — `.format_messages()` te muestra exactamente qué va al LLM
4. Hay **múltiples tipos** — `from_messages`, `from_template`, `MessagesPlaceholder`, composición con `+`
5. Los **valores parciales** permiten fijar parámetros y dejar solo los variables libres

### Próximos pasos

- **E03**: conectar `ChatPromptTemplate` con `ChatOpenAI` usando LCEL (`prompt | llm | parser`)
- **E04**: agregar memoria con `MessagesPlaceholder` y `RunnableWithMessageHistory`
- **E10**: usar el template en un pipeline RAG completo